# 03 - Model Architecture

A **VGG-style CNN trained from scratch** for 15-class vegetable classification — this is exactly the architecture saved in `models/vegetable_cnn.h5`:

| Block | Layer stack | Output |
|---|---|---|
| 1 | Conv 5×5/32 → Conv 5×5/32 → BatchNorm → MaxPool → Dropout 25% | 71 × 71 |
| 2 | Conv 3×3/64 → Conv 3×3/64 → BatchNorm → MaxPool → Dropout 25% | 33 × 33 |
| Head | Flatten → Dense 256 (ReLU) → Dropout 50% → Dense 15 (softmax) | 15 |

Input: `150 × 150 × 3` raw pixels. The definition lives in `src/models.py` so the Kaggle notebook and any retraining shares the same source of truth.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import tensorflow as tf

from src.config import IMG_SIZE, NUM_CLASSES  # noqa: E402
from src.models import build_vegetable_cnn, compile_vegetable_cnn  # noqa: E402

print(f"TensorFlow {tf.__version__}")


I0000 00:00:1790237024.181161 1661245 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790237024.346590 1661245 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790237032.329992 1661245 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow 2.21.0


In [2]:

model = build_vegetable_cnn()
model.summary()


/mnt/hdd/projects/Vegetables-Classification/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1790237039.508250 1661245 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "vegetable_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 146, 146, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 142, 142, 32)   │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 142, 142, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 71, 71, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 71, 71, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 69, 69, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 67, 67, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 67, 67, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 33, 33, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 33, 33, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 69696)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    17,842,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,930,159 (68.40 MB)

 Trainable params: 17,929,967 (68.40 MB)

 Non-trainable params: 192 (768.00 B)

## Parameter accounting

In [3]:

trainable = model.count_params()
non_trainable = int(sum(int(w.shape.num_elements()) for w in model.non_trainable_weights))
total = trainable + non_trainable
size_mb = total * 4 / (1024 ** 2)  # float32

print(f"Trainable params     : {trainable:>10,}")
print(f"Non-trainable params : {non_trainable:>10,}  (BN running mean/var)")
print(f"Total params         : {total:>10,}")
print(f"Model size (float32) : {size_mb:8.1f} MB")

print("\nPer layer:")
for layer in model.layers:
    n = int(sum(int(w.shape.num_elements()) for w in layer.weights))
    if n:
        print(f"  {layer.name:<22s} params: {n:>10,}")


Trainable params     : 17,930,159
Non-trainable params :        192  (BN running mean/var)
Total params         : 17,930,351
Model size (float32) :     68.4 MB

Per layer:
  conv2d                 params:      2,432
  conv2d_1               params:     25,632
  batch_normalization    params:        128
  conv2d_2               params:     18,496
  conv2d_3               params:     36,928
  batch_normalization_1  params:        256
  dense                  params: 17,842,432
  dense_1                params:      3,855


## Sanity check: forward pass

In [4]:

x = tf.random.uniform((2, *IMG_SIZE, 3), 0.0, 255.0)  # raw pixels, like training
pred = model(x, training=False)
print("input shape :", x.shape)
print("output shape:", pred.shape)
print("rows sum to 1 (softmax):", np.allclose(pred.numpy().sum(axis=1), 1.0))
print("argmax class indices    :", pred.numpy().argmax(axis=1))

model = compile_vegetable_cnn(model)
print("\nCompiled with Adam(1e-3) + categorical_crossentropy.")


input shape : (2, 150, 150, 3)
output shape: (2, 15)
rows sum to 1 (softmax): True
argmax class indices    : [2 2]

Compiled with Adam(1e-3) + categorical_crossentropy.


## Design notes
- Filters grow (32 → 64) while space halves, balancing parameters and activations.
- BatchNorm stabilises training and lets the model learn its own input scaling (why raw `[0, 255]` input works).
- Dropout between blocks and before the head regularises the ~17.9M-parameter network.

**Training happens in `04_train_on_kaggle.ipynb`; this notebook never fits the model.**